# DIP Assignment 2 — Question 5

## Denoising and the Wiener Filter

This notebook implements Sections 5.1 and 5.2 independently. It preserves the
supplied 3072×2048 RGB reference image, estimates the signal PSD from each
noisy input rather than from the clean reference, implements the frequency
domain Wiener estimator with NumPy, compares it with Gaussian, Butterworth,
and bilateral denoisers, tests noise-power misestimation, and leaves a
ground-truth-free real-photo workflow ready for a separately supplied image.

All filtering required by the assignment is implemented with NumPy array
arithmetic and `numpy.fft`. Display thumbnails are decimated copies only;
the computational arrays retain their original dimensions.


## Official requirements and conventions

The official Question 5 model is
\[
g(x,y)=f(x,y)+n(x,y),
\]
with uncorrelated, wide-sense stationary clean signal and additive noise.
The controlled image is normalized by 255, the injected noise levels are
\(\sigma\in\{5,10,20,40\}\) 8-bit levels, and all methods receive the same
unclipped noisy realization at each level. Metrics use clipped copies in
\([0,1]\), data range \(L=1\), and channel-wise RGB aggregation.

The primary signal PSD estimate is a radially averaged noisy periodogram,
with the known white-noise floor subtracted and the radial profile smoothed
by a centered NumPy moving average. This is a practical isotropic estimate,
not a claim that photographic texture is directionally isotropic.


In [1]:

import csv
import gc
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.set_printoptions(precision=6, suppress=True)
plt.rcParams["figure.dpi"] = 100
ROOT = Path.cwd()
DATA = ROOT / "A2" / "Q5,6"
if not DATA.exists():
    hits = list(ROOT.rglob("reference_image.jpg"))
    if not hits:
        raise FileNotFoundError("Could not locate A2/Q5,6/reference_image.jpg.")
    DATA = hits[0].parent
image_path = DATA / "reference_image.jpg"
if not image_path.exists():
    raise FileNotFoundError(f"Reference image not found: {image_path}")
OUT = ROOT / "outputs" / "Q5"
FIG, TAB, NOISY, DENOISED, REAL, VER, META = [
    OUT / x for x in ["figures", "tables", "noisy_images", "denoised_images",
                      "real_photo", "verification", "metadata"]
]
for p in [FIG, TAB, NOISY, DENOISED, REAL, VER, META]:
    p.mkdir(parents=True, exist_ok=True)

def exact_cpu_model():
    """Return the OS-reported CPU model, with a platform fallback."""
    try:
        result = subprocess.check_output(
            ["powershell", "-NoProfile", "-Command",
             "(Get-CimInstance Win32_Processor | Select-Object -First 1 -ExpandProperty Name)"],
            text=True, stderr=subprocess.DEVNULL).strip()
        return result or platform.processor() or "not reported"
    except Exception:
        return platform.processor() or "not reported"

def savefig(name):
    """Save and close a figure under the Q5 figures directory."""
    path = FIG / name
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.close()
    return path

def write_rows(name, rows):
    """Write a nonempty list of dictionaries as a CSV table."""
    if not rows:
        raise ValueError("Cannot write an empty table.")
    with open(TAB / name, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

reference_u8 = np.asarray(Image.open(image_path).convert("RGB"))
if reference_u8.shape != (2048, 3072, 3) or reference_u8.dtype != np.uint8:
    raise ValueError(f"Unexpected reference properties: {reference_u8.shape}, {reference_u8.dtype}")
clean = reference_u8.astype(np.float32) / 255.0
metadata = {
    "reference path": str(image_path),
    "shape": str(reference_u8.shape),
    "mode": "RGB",
    "dtype": str(reference_u8.dtype),
    "min": int(reference_u8.min()),
    "max": int(reference_u8.max()),
    "dynamic range": int(reference_u8.max() - reference_u8.min()),
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "os": platform.platform(),
    "cpu": exact_cpu_model(),
    "total RAM bytes": int(__import__("psutil").virtual_memory().total),
}
write_rows("00_metadata.csv", [metadata])
(META / "experiment_settings.txt").write_text(
    "seed=20261029\nnoise_levels_8bit=5,10,20,40\n"
    "PSD=radial periodogram minus known white floor, moving average width 9\n"
    "metrics=clipped to [0,1], RGB channel mean SSIM, data range 1\n",
    encoding="utf-8")
print("Environment and reference metadata:", metadata)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(clean[::8, ::8]); ax[0].set_title("Reference RGB (display thumbnail)"); ax[0].axis("off")
ax[1].imshow(clean[::8, ::8, 0], cmap="gray"); ax[1].set_title("Red channel thumbnail"); ax[1].axis("off")
savefig("01_reference_and_channel.png")
print("Both dimensions exceed 1024:", reference_u8.shape[0] > 1024 and reference_u8.shape[1] > 1024)


Environment and reference metadata: {'reference path': 'D:\\College\\IIITH\\DIP\\Assignment-2\\DIP_A2\\A2\\Q5,6\\reference_image.jpg', 'shape': '(2048, 3072, 3)', 'mode': 'RGB', 'dtype': 'uint8', 'min': 0, 'max': 255, 'dynamic range': 255, 'python': '3.13.14', 'numpy': '2.2.6', 'os': 'Windows-11-10.0.26300-SP0', 'cpu': '13th Gen Intel(R) Core(TM) i7-13700HX', 'total RAM bytes': 16886128640}
Both dimensions exceed 1024: True


## RGB normalization and controlled noise design

The normalized standard deviation is \(\sigma_\mathrm{norm}=\sigma/255\);
the variance, which is the white-noise PSD under this periodogram
normalization, is \(\sigma_\mathrm{norm}^2\). Standard deviation measures a
typical amplitude while variance is its squared power. One standard-normal
RGB field is scaled to each requested level, so every method sees exactly
the same noisy array at a given level.


In [2]:

SEED = 20261029
NOISE_LEVELS = [5, 10, 20, 40]
rng = np.random.default_rng(SEED)
standard_noise = rng.standard_normal(clean.shape).astype(np.float32)
noise_rows = []
noisy_inputs = {}
for sigma8 in NOISE_LEVELS:
    sigma_norm = sigma8 / 255.0
    noisy_inputs[sigma8] = clean + sigma_norm * standard_noise
    noise_rows.append({"sigma 8-bit": sigma8, "sigma normalized": sigma_norm,
                       "noise variance / PSD": sigma_norm**2,
                       "seed": SEED, "clipped before filtering": False})
write_rows("01_noise_parameters.csv", noise_rows)
for sigma8 in NOISE_LEVELS:
    display = np.clip(noisy_inputs[sigma8][::8, ::8], 0, 1)
    Image.fromarray(np.round(display * 255).astype(np.uint8)).save(
        NOISY / f"noisy_sigma_{sigma8}.png")
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, sigma8 in zip(ax, NOISE_LEVELS):
    a.imshow(np.clip(noisy_inputs[sigma8][::8, ::8], 0, 1))
    a.set_title(f"noisy sigma={sigma8}"); a.axis("off")
savefig("02_noisy_levels.png")
print("Noise parameter table:")
for row in noise_rows:
    print(row)


Noise parameter table:
{'sigma 8-bit': 5, 'sigma normalized': 0.0196078431372549, 'noise variance / PSD': 0.00038446751249519417, 'seed': 20261029, 'clipped before filtering': False}
{'sigma 8-bit': 10, 'sigma normalized': 0.0392156862745098, 'noise variance / PSD': 0.0015378700499807767, 'seed': 20261029, 'clipped before filtering': False}
{'sigma 8-bit': 20, 'sigma normalized': 0.0784313725490196, 'noise variance / PSD': 0.006151480199923107, 'seed': 20261029, 'clipped before filtering': False}
{'sigma 8-bit': 40, 'sigma normalized': 0.1568627450980392, 'noise variance / PSD': 0.024605920799692427, 'seed': 20261029, 'clipped before filtering': False}


## 5.1.1 Statistical model and 5.1.2 linear-MMSE derivation

Wide-sense stationarity means that the mean is constant and the
autocorrelation depends on displacement, so a power spectral density
describes second-order statistics. Uncorrelated signal and noise have zero
cross-correlation, which removes the cross terms below. These assumptions
make the frequency-by-frequency optimization useful, but a natural
photograph has edges, regions, color correlations, and spatially varying
noise, so stationarity is only an approximation.

Let
\[
G=F+N,\qquad \widehat F=WG,\qquad E=F-WG.
\]
The per-frequency error power is
\[
J(W)=\mathbb E|F-WG|^2
 =S_{ff}-W S_{gf}-W^*S_{fg}+|W|^2S_{gg}.
\]
Using Wirtinger differentiation with respect to \(W^*\),
\[
\frac{\partial J}{\partial W^*}=-S_{fg}+WS_{gg}=0,
\qquad W=\frac{S_{fg}}{S_{gg}}.
\]
Because \(G=F+N\), signal-noise uncorrelatedness gives
\[
S_{fg}=E[FG^*]=S_{ff},\qquad
S_{gg}=E[GG^*]=S_{ff}+S_{nn}.
\]
Therefore
\[
\boxed{W(u,v)=\frac{S_{ff}(u,v)}{S_{ff}(u,v)+S_{nn}(u,v)}},
\qquad
\boxed{\widehat F(u,v)=W(u,v)G(u,v)}.
\]
Here \(S_{ff}\) and \(S_{nn}\) are clean-signal and noise PSDs, and \(W\) is
the frequency-dependent gain. It approaches one when signal power dominates
and zero when noise dominates. This is optimal among linear estimators under
the stated second-order model; it is not a universal optimality claim over
nonlinear estimators.


## 5.1.3–5.1.5 PSD estimation and from-scratch Wiener implementation

For an \(M\times N\) channel I use
\[
P_g(u,v)=|G(u,v)|^2/(MN).
\]
White noise with variance \(\sigma_n^2\) has expected spectral floor
\(\sigma_n^2\). The radial average is formed on the rectangular FFT grid,
smoothed by a centered width-9 moving average, and mapped back to every
frequency bin. The estimate is
\[
\widehat S_{ff}=\max(\operatorname{smooth}(\overline{P_g})-\sigma_n^2,\epsilon).
\]
The estimator is intentionally based only on the noisy channel. Radial
averaging reduces variance but suppresses directional texture information,
and subtraction becomes uncertain at high noise.


In [3]:

def radial_frequency_grid(shape):
    """Return rectangular cycles-per-pixel radius and integer radial bins."""
    fy = np.fft.fftfreq(shape[0])[:, None]
    fx = np.fft.fftfreq(shape[1])[None, :]
    radius = np.sqrt(fx * fx + fy * fy)
    bins = np.rint(radius * max(shape)).astype(np.int32)
    return radius, bins

def moving_average(values, width=9):
    """Smooth a one-dimensional profile with a centered NumPy box average."""
    if width < 1 or width % 2 == 0:
        raise ValueError("width must be a positive odd integer")
    kernel = np.ones(width, dtype=np.float64) / width
    padded = np.pad(values, width // 2, mode="edge")
    return np.convolve(padded, kernel, mode="valid")

def estimate_signal_psd(noisy_channel, noise_power, bins, smoothing_width=9):
    """Estimate nonnegative signal PSD from a noisy periodogram.

    The periodogram is |FFT(channel)|^2/(M*N). Radial bin means are smoothed
    with a centered box window, the known white-noise floor is subtracted,
    and an epsilon floor prevents invalid Wiener denominators.
    """
    a = np.asarray(noisy_channel, dtype=np.float32)
    if a.ndim != 2:
        raise ValueError("noisy_channel must be two-dimensional")
    F = np.fft.fft2(a)
    periodogram = (np.abs(F) ** 2 / a.size).astype(np.float32)
    max_bin = int(bins.max())
    sums = np.bincount(bins.ravel(), weights=periodogram.ravel(), minlength=max_bin + 1)
    counts = np.bincount(bins.ravel(), minlength=max_bin + 1)
    radial_noisy = sums / np.maximum(counts, 1)
    radial_smooth = moving_average(radial_noisy, smoothing_width)
    signal_radial = np.maximum(radial_smooth - noise_power, np.finfo(np.float32).eps)
    signal_psd = signal_radial[bins]
    return signal_psd.astype(np.float32), radial_noisy, radial_smooth, periodogram

def wiener_denoise_channel(noisy_channel, noise_power, bins, smoothing_width=9,
                           fixed_signal_psd=None):
    """Apply a channel-wise frequency-domain Wiener estimator from scratch."""
    a = np.asarray(noisy_channel, dtype=np.float32)
    if fixed_signal_psd is None:
        signal_psd, radial_noisy, radial_smooth, periodogram = estimate_signal_psd(
            a, noise_power, bins, smoothing_width)
    else:
        signal_psd = np.asarray(fixed_signal_psd, dtype=np.float32)
        radial_noisy = radial_smooth = periodogram = None
    denominator = np.maximum(signal_psd + noise_power, np.finfo(np.float32).eps)
    gain = np.clip(signal_psd / denominator, 0.0, 1.0)
    F = np.fft.fft2(a)
    result_complex = np.fft.ifft2(gain * F)
    result = result_complex.real.astype(np.float32)
    return result, gain, signal_psd, (radial_noisy, radial_smooth, periodogram), float(np.max(np.abs(result_complex.imag)))

def wiener_denoise_rgb(noisy_rgb, noise_power, bins, smoothing_width=9,
                       fixed_signal_psds=None):
    """Apply the Wiener estimator independently to the three RGB channels."""
    a = np.asarray(noisy_rgb, dtype=np.float32)
    if a.ndim != 3 or a.shape[2] != 3:
        raise ValueError("noisy_rgb must have shape (M,N,3)")
    output = np.empty_like(a)
    gains = []
    psds = []
    details = []
    imag_residuals = []
    for c in range(3):
        fixed = None if fixed_signal_psds is None else fixed_signal_psds[c]
        result, gain, psd, detail, imag = wiener_denoise_channel(
            a[:, :, c], noise_power, bins, smoothing_width, fixed)
        output[:, :, c] = result
        gains.append(gain); psds.append(psd); details.append(detail)
        imag_residuals.append(imag)
    return output, gains, psds, details, imag_residuals

radius, radial_bins = radial_frequency_grid(clean.shape[:2])
print("Reusable frequency grid:", radius.shape, "maximum radial bin:", int(radial_bins.max()))


Reusable frequency grid: (2048, 3072) maximum radial bin: 2172


## Reference Gaussian, Butterworth, bilateral, PSNR, and SSIM implementations

The Gaussian and Butterworth filters use the original rectangular dimensions
and circular FFT boundary convention. Their cutoffs are fixed in cycles per
pixel: `D0=0.08`, with a fourth-order Butterworth. The bilateral reference is
the Question 4 NumPy algorithm applied independently to RGB channels, using
reflected boundaries, `sigma_s=1.5`, and `sigma_r=2*sigma_norm`; it does not
use joint RGB distances.

All metrics clip only their metric copies to `[0,1]`. SSIM uses an 11×11
separable Gaussian window with sigma 1.5, `K1=.01`, `K2=.03`, `L=1`, and
reflective boundaries. Scores are averaged across the three channels.


In [4]:

GAUSSIAN_D0 = 0.08
BUTTERWORTH_D0 = 0.08
BUTTERWORTH_ORDER = 4
def frequency_lowpass_channel(channel, kind, d0=0.08, order=4):
    """Apply a rectangular Gaussian or Butterworth low-pass with NumPy FFT."""
    a = np.asarray(channel, dtype=np.float32)
    fy = np.fft.fftfreq(a.shape[0])[:, None]
    fx = np.fft.fftfreq(a.shape[1])[None, :]
    d = np.sqrt(fx * fx + fy * fy)
    if kind == "gaussian":
        H = np.exp(-(d * d) / (2 * d0 * d0))
    elif kind == "butterworth":
        H = 1.0 / (1.0 + (d / d0) ** (2 * order))
    else:
        raise ValueError("kind must be gaussian or butterworth")
    return np.fft.ifft2(np.fft.fft2(a) * H).real.astype(np.float32)

def bilateral_channel(channel, sigma_s, sigma_r):
    """Apply reflected-boundary bilateral filtering using NumPy shifts."""
    a = np.asarray(channel, dtype=np.float32)
    radius_i = max(1, int(np.ceil(2 * sigma_s)))
    pad = np.pad(a, radius_i, mode="reflect")
    yy, xx = np.mgrid[-radius_i:radius_i+1, -radius_i:radius_i+1]
    spatial = np.exp(-(xx * xx + yy * yy) / (2 * sigma_s * sigma_s))
    numerator = np.zeros_like(a, dtype=np.float32)
    denominator = np.zeros_like(a, dtype=np.float32)
    for dy, dx in zip(yy.ravel(), xx.ravel()):
        patch = pad[radius_i + dy:radius_i + dy + a.shape[0],
                    radius_i + dx:radius_i + dx + a.shape[1]]
        weights = spatial[dy + radius_i, dx + radius_i] * np.exp(
            -((patch - a) ** 2) / (2 * sigma_r * sigma_r))
        numerator += weights * patch
        denominator += weights
    return numerator / np.maximum(denominator, np.finfo(np.float32).eps)

def apply_rgb_filter(image, method, sigma_norm):
    """Apply one declared RGB reference method channel by channel."""
    out = np.empty_like(image, dtype=np.float32)
    for c in range(3):
        if method == "gaussian":
            out[:, :, c] = frequency_lowpass_channel(image[:, :, c], "gaussian", GAUSSIAN_D0)
        elif method == "butterworth":
            out[:, :, c] = frequency_lowpass_channel(
                image[:, :, c], "butterworth", BUTTERWORTH_D0, BUTTERWORTH_ORDER)
        elif method == "bilateral":
            out[:, :, c] = bilateral_channel(image[:, :, c], 1.5, 2 * sigma_norm)
        else:
            raise ValueError(method)
    return out

def psnr_rgb(reference, estimate):
    """Return normalized RGB PSNR after consistent metric clipping."""
    a = np.clip(np.asarray(reference, dtype=np.float32), 0, 1)
    b = np.clip(np.asarray(estimate, dtype=np.float32), 0, 1)
    mse = float(np.mean((a - b) ** 2))
    return float("inf") if mse == 0 else float(10 * np.log10(1.0 / mse))

def _separable_filter(a, kernel):
    """Apply a small separable kernel with reflective boundaries."""
    radius_i = len(kernel) // 2
    p = np.pad(a, ((radius_i, radius_i), (0, 0)), mode="reflect")
    vertical = sum(float(kernel[i]) * p[i:i+a.shape[0]] for i in range(len(kernel)))
    p = np.pad(vertical, ((0, 0), (radius_i, radius_i)), mode="reflect")
    return sum(float(kernel[i]) * p[:, i:i+a.shape[1]] for i in range(len(kernel)))

def ssim_channel(reference, estimate):
    """Compute SSIM with the documented 11x11 Gaussian/reflect convention."""
    a = np.clip(np.asarray(reference, dtype=np.float32), 0, 1)
    b = np.clip(np.asarray(estimate, dtype=np.float32), 0, 1)
    x = np.arange(-5, 6, dtype=np.float32)
    kernel = np.exp(-(x * x) / (2 * 1.5 * 1.5)); kernel /= kernel.sum()
    mu_a = _separable_filter(a, kernel); mu_b = _separable_filter(b, kernel)
    var_a = _separable_filter(a * a, kernel) - mu_a * mu_a
    var_b = _separable_filter(b * b, kernel) - mu_b * mu_b
    cov = _separable_filter(a * b, kernel) - mu_a * mu_b
    c1, c2 = 0.01**2, 0.03**2
    score = ((2 * mu_a * mu_b + c1) * (2 * cov + c2) /
             ((mu_a * mu_a + mu_b * mu_b + c1) * (var_a + var_b + c2)))
    return float(np.mean(score))

def ssim_rgb(reference, estimate):
    """Return the arithmetic mean of the three channel SSIM scores."""
    return float(np.mean([ssim_channel(reference[:, :, c], estimate[:, :, c])
                          for c in range(3)]))

filter_rows = [
    {"method": "wiener", "parameters": "radial PSD, width=9, known white floor",
     "frequency dimensions": str(clean.shape[:2]), "boundary": "periodic FFT",
     "parameter selection": "fixed/predeclared", "oracle tuned": False},
    {"method": "gaussian", "parameters": f"D0={GAUSSIAN_D0} cycles/pixel",
     "frequency dimensions": str(clean.shape[:2]), "boundary": "periodic FFT",
     "parameter selection": "fixed/predeclared", "oracle tuned": False},
    {"method": "butterworth", "parameters": f"D0={BUTTERWORTH_D0}, n={BUTTERWORTH_ORDER}",
     "frequency dimensions": str(clean.shape[:2]), "boundary": "periodic FFT",
     "parameter selection": "fixed/predeclared", "oracle tuned": False},
    {"method": "bilateral", "parameters": "sigma_s=1.5, sigma_r=2*sigma_norm, reflect",
     "frequency dimensions": str(clean.shape[:2]), "boundary": "reflect",
     "parameter selection": "fixed/predeclared", "oracle tuned": False},
]
write_rows("02_filter_settings.csv", filter_rows)


## 5.1.6–5.1.11 Comparative experiments

For every noise level, the Wiener, Gaussian, Butterworth, and bilateral
outputs below use the same raw noisy RGB input. The Wiener PSD is estimated
independently for each noisy channel. The fixed Gaussian and Butterworth
responses are not adaptive; the Wiener response is adaptive through the
estimated radial signal power. A nonlinear bilateral filter can therefore
win on an image or metric even though the Wiener estimator is the linear-MMSE
solution under its model.


In [5]:

comparison_rows = []
gain_summaries = {}
psd_examples = {}
outputs = {}
for sigma8 in NOISE_LEVELS:
    sigma_norm = sigma8 / 255.0
    noisy = noisy_inputs[sigma8]
    t0 = time.perf_counter()
    wiener, gains, psds, details, imag_residuals = wiener_denoise_rgb(
        noisy, sigma_norm**2, radial_bins, smoothing_width=9)
    outputs[(sigma8, "wiener")] = wiener
    outputs[(sigma8, "gaussian")] = apply_rgb_filter(noisy, "gaussian", sigma_norm)
    outputs[(sigma8, "butterworth")] = apply_rgb_filter(noisy, "butterworth", sigma_norm)
    outputs[(sigma8, "bilateral")] = apply_rgb_filter(noisy, "bilateral", sigma_norm)
    outputs[(sigma8, "noisy")] = noisy
    for method in ["noisy", "wiener", "gaussian", "butterworth", "bilateral"]:
        z = outputs[(sigma8, method)]
        comparison_rows.append({
            "sigma 8-bit": sigma8, "method": method,
            "PSNR dB": psnr_rgb(clean, z),
            "SSIM": ssim_rgb(clean, z),
            "parameter selection": "baseline" if method == "noisy" else "fixed/predeclared",
            "oracle tuned": False,
            "metric clipping": "[0,1]", "data range": 1,
        })
    gain_summaries[sigma8] = {
        "min gain": float(min(np.min(g) for g in gains)),
        "max gain": float(max(np.max(g) for g in gains)),
        "mean gain": float(np.mean([g.mean() for g in gains])),
        "max imaginary residual": max(imag_residuals),
        "seconds": time.perf_counter() - t0,
    }
    if sigma8 in [5, 40]:
        psd_examples[sigma8] = (details, gains, psds)
    for method in ["wiener", "gaussian", "butterworth", "bilateral"]:
        display = np.clip(outputs[(sigma8, method)][::8, ::8], 0, 1)
        Image.fromarray(np.round(display * 255).astype(np.uint8)).save(
            DENOISED / f"{method}_sigma_{sigma8}.png")
    Image.fromarray(np.round(np.clip(noisy[::8, ::8], 0, 1) * 255).astype(np.uint8)).save(
        NOISY / f"noisy_sigma_{sigma8}.png")
    print("Completed sigma", sigma8, "gain summary:", gain_summaries[sigma8])
    gc.collect()

write_rows("03_psnr_ssim_comparison.csv", comparison_rows)
summary_rows = []
for sigma8 in NOISE_LEVELS:
    subset = [r for r in comparison_rows if r["sigma 8-bit"] == sigma8 and r["method"] != "noisy"]
    best_psnr = max(subset, key=lambda r: r["PSNR dB"])
    best_ssim = max(subset, key=lambda r: r["SSIM"])
    summary_rows.append({"sigma 8-bit": sigma8, "best PSNR method": best_psnr["method"],
                         "best PSNR dB": best_psnr["PSNR dB"],
                         "best SSIM method": best_ssim["method"],
                         "best SSIM": best_ssim["SSIM"]})
write_rows("04_best_method_summary.csv", summary_rows)
print("PSNR/SSIM comparison table:")
for row in comparison_rows: print(row)
print("Best-method summary:")
for row in summary_rows: print(row)


Completed sigma 5 gain summary: {'min gain': 0.029738517478108406, 'max gain': 1.0, 'mean gain': 0.45143356919288635, 'max imaginary residual': 2.0484402796228096e-07, 'seconds': 54.527720600017346}


Completed sigma 10 gain summary: {'min gain': 7.750983058940619e-05, 'max gain': 1.0, 'mean gain': 0.26381775736808777, 'max imaginary residual': 1.9881144908140413e-07, 'seconds': 53.61110990002635}


Completed sigma 20 gain summary: {'min gain': 1.9378585420781747e-05, 'max gain': 1.0, 'mean gain': 0.13990654051303864, 'max imaginary residual': 1.870854475782835e-07, 'seconds': 53.5617127000005}


Completed sigma 40 gain summary: {'min gain': 4.844716386287473e-06, 'max gain': 1.0, 'mean gain': 0.06606508046388626, 'max imaginary residual': 1.9083267943642568e-07, 'seconds': 52.327039499999955}
PSNR/SSIM comparison table:
{'sigma 8-bit': 5, 'method': 'noisy', 'PSNR dB': 34.28953420724341, 'SSIM': 0.8414936463038126, 'parameter selection': 'baseline', 'oracle tuned': False, 'metric clipping': '[0,1]', 'data range': 1}
{'sigma 8-bit': 5, 'method': 'wiener', 'PSNR dB': 37.66605610515175, 'SSIM': 0.9337671001752218, 'parameter selection': 'fixed/predeclared', 'oracle tuned': False, 'metric clipping': '[0,1]', 'data range': 1}
{'sigma 8-bit': 5, 'method': 'gaussian', 'PSNR dB': 29.296327090567434, 'SSIM': 0.8037162621816, 'parameter selection': 'fixed/predeclared', 'oracle tuned': False, 'metric clipping': '[0,1]', 'data range': 1}
{'sigma 8-bit': 5, 'method': 'butterworth', 'PSNR dB': 27.99629982344898, 'SSIM': 0.7376550634702047, 'parameter selection': 'fixed/predeclared', 'oracle 

In [6]:

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
methods = ["wiener", "gaussian", "butterworth", "bilateral"]
for method in methods:
    rows_m = [r for r in comparison_rows if r["method"] == method]
    ax[0].plot(NOISE_LEVELS, [r["PSNR dB"] for r in rows_m], marker="o", label=method)
    ax[1].plot(NOISE_LEVELS, [r["SSIM"] for r in rows_m], marker="o", label=method)
ax[0].set(xlabel="noise sigma (8-bit)", ylabel="PSNR (dB)", title="PSNR versus noise")
ax[1].set(xlabel="noise sigma (8-bit)", ylabel="mean RGB SSIM", title="SSIM versus noise")
for a in ax: a.legend()
savefig("03_psnr_ssim_curves.png")
fig, ax = plt.subplots(2, 3, figsize=(12, 7))
for row_i, sigma8 in enumerate([5, 40]):
    details, gains, psds = psd_examples[sigma8]
    radial_x = np.arange(len(details[0][1]))
    for c in range(3):
        radial_noisy = details[c][1]; radial_smooth = details[c][2]
        ax[row_i, c].plot(radial_x[:300], radial_noisy[:300], alpha=.5, label="noisy periodogram")
        ax[row_i, c].plot(radial_x[:300], radial_smooth[:300], label="smoothed")
        ax[row_i, c].axhline((sigma8/255)**2, color="k", ls="--", label="white floor")
        ax[row_i, c].set_title(f"sigma={sigma8}, channel={c}")
        ax[row_i, c].set_xlabel("radial bin"); ax[row_i, c].set_ylabel("power")
        ax[row_i, c].legend(fontsize=7)
savefig("04_psd_estimation_examples.png")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, sigma8 in zip(ax, [5, 40]):
    for c, gain in enumerate(psd_examples[sigma8][1]):
        a.plot(np.sort(radius.ravel())[:400], np.sort(gain.ravel())[:400], label=f"channel {c}")
    a.set_title(f"Wiener gain distributions, sigma={sigma8}")
    a.set_xlabel("sorted frequency radius"); a.set_ylabel("gain"); a.legend()
savefig("05_wiener_gain_examples.png")
for sigma8 in [5, 40]:
    fig, ax = plt.subplots(2, 3, figsize=(12, 7))
    for j, method in enumerate(["noisy", "wiener", "gaussian", "butterworth", "bilateral"]):
        ax.flat[j].imshow(np.clip(outputs[(sigma8, method)][::8, ::8], 0, 1))
        ax.flat[j].set_title(method); ax.flat[j].axis("off")
    ax.flat[5].axis("off")
    savefig(f"06_comparison_grid_sigma_{sigma8}.png")


C:\Users\Acer\AppData\Local\Temp\ipykernel_52672\760315608.py:47: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(path, dpi=130, bbox_inches="tight")


### Measured interpretation

The printed tables are the executed evidence. As noise increases, the
estimated white floor grows and the Wiener gain generally shrinks over more
frequencies, but PSD subtraction also becomes less certain. Gaussian
attenuation is smooth and predetermined; Butterworth has an order-dependent
transition; Wiener attenuation follows the estimated signal spectrum.
PSNR and SSIM can select different winners because PSNR is global squared
error while SSIM emphasizes local luminance, contrast, and structure.


## 5.2.1 Controlled noise-power misestimation at σ=20

The signal PSD is held fixed at the correctly specified σ=20 estimate. Only
the assumed noise power in
\[
W=\widehat S_{ff}/(\widehat S_{ff}+S_{nn,\mathrm{assumed}})
\]
changes. Thus half the true power retains more noise, while twice the true
power suppresses more signal detail. The three outputs use the identical
σ=20 noisy input.


In [7]:

sigma20 = 20
noisy20 = noisy_inputs[sigma20]
true_power20 = (sigma20 / 255.0) ** 2
baseline20, gains20, psds20, details20, imag20 = wiener_denoise_rgb(
    noisy20, true_power20, radial_bins, smoothing_width=9)
mis_rows = []
mis_outputs = {}
for ratio in [0.5, 1.0, 2.0]:
    assumed = ratio * true_power20
    z, gains_m, _, _, imag_m = wiener_denoise_rgb(
        noisy20, assumed, radial_bins, smoothing_width=9, fixed_signal_psds=psds20)
    mis_outputs[ratio] = z
    mis_rows.append({"assumed noise power": assumed, "ratio to true": ratio,
                     "min gain": float(min(g.min() for g in gains_m)),
                     "max gain": float(max(g.max() for g in gains_m)),
                     "mean gain": float(np.mean([g.mean() for g in gains_m])),
                     "PSNR dB": psnr_rgb(clean, z), "SSIM": ssim_rgb(clean, z),
                     "same noisy input": True, "signal PSD held fixed": True,
                     "observation": "more noise retained" if ratio < 1 else
                     ("baseline" if ratio == 1 else "more signal suppression")})
write_rows("05_noise_power_misestimation.csv", mis_rows)
print("Noise-power misestimation table:")
for row in mis_rows: print(row)
fig, ax = plt.subplots(1, 5, figsize=(18, 4))
for a, z, title in zip(ax,
        [clean, noisy20, mis_outputs[.5], mis_outputs[1.0], mis_outputs[2.0]],
        ["clean", "noisy sigma=20", "0.5 true power", "true power", "2 true power"]):
    a.imshow(np.clip(z[::8, ::8], 0, 1)); a.set_title(title); a.axis("off")
savefig("07_noise_power_misestimation.png")


Noise-power misestimation table:
{'assumed noise power': 0.0030757400999615533, 'ratio to true': 0.5, 'min gain': 3.8756417779950425e-05, 'max gain': 1.0, 'mean gain': 0.19502170383930206, 'PSNR dB': 30.469603768712545, 'SSIM': 0.7550527652104696, 'same noisy input': True, 'signal PSD held fixed': True, 'observation': 'more noise retained'}
{'assumed noise power': 0.006151480199923107, 'ratio to true': 1.0, 'min gain': 1.9378585420781747e-05, 'max gain': 1.0, 'mean gain': 0.13990654051303864, 'PSNR dB': 30.730363373401858, 'SSIM': 0.7920261224110922, 'same noisy input': True, 'signal PSD held fixed': True, 'observation': 'baseline'}
{'assumed noise power': 0.012302960399846213, 'ratio to true': 2.0, 'min gain': 9.689386388345156e-06, 'max gain': 1.0, 'mean gain': 0.09723695367574692, 'PSNR dB': 30.36933743273532, 'SSIM': 0.8076342145601908, 'same noisy input': True, 'signal PSD held fixed': True, 'observation': 'more signal suppression'}


WindowsPath('D:/College/IIITH/DIP/Assignment-2/DIP_A2/outputs/Q5/figures/07_noise_power_misestimation.png')

## 5.2.2 Why a correctly specified Wiener filter can blur edges

The Wiener estimator is linear and uses stationary second-order power, not
an explicit edge map. A sharp edge is represented by high-frequency
components, and those frequencies can have power comparable to the noise.
Even with the correct noise variance, the gain attenuates some of those
components, so the transition broadens. MMSE minimizes expected squared
error under the model; it does not directly optimize edge sharpness or
perceptual preference. A bilateral filter can preserve a particular edge by
using local intensity differences, but that nonlinear behavior is not
universally superior and can fail under other textures or noise models.
The fixed-noise experiment above provides the measured gain and metric
evidence for this trade-off.


## 5.2.3 Real noisy photograph case study

The supplied archive contains no personal photograph, so this section does
not fabricate a result. When `data/real_noisy_photo.jpg` is supplied, the
cell below loads it, records its properties, selects the declared flat
region, estimates noise from robust neighboring differences, applies the
same white-noise Wiener implementation, and saves qualitative crops and
residuals. No PSNR or SSIM is reported without an independently registered
clean reference.


In [8]:

real_photo_path = ROOT / "data" / "real_noisy_photo.jpg"
real_rows = []
if not real_photo_path.exists():
    print("Real-photo case study pending: data/real_noisy_photo.jpg was not supplied.")
    real_rows.append({"status": "pending", "path": str(real_photo_path),
                      "PSNR/SSIM": "not reported without clean registered reference"})
else:
    real_u8 = np.asarray(Image.open(real_photo_path).convert("RGB"))
    real = real_u8.astype(np.float32) / 255.0
    h, w = real.shape[:2]
    y0, y1 = max(0, h//10), min(h, h//3)
    x0, x1 = max(0, w//10), min(w, w//3)
    flat = real[y0:y1, x0:x1]
    estimates = []
    for c in range(3):
        d = np.concatenate([np.diff(flat[:, :, c], axis=0).ravel(),
                            np.diff(flat[:, :, c], axis=1).ravel()])
        centered = d - np.median(d)
        estimates.append(float(np.median(np.abs(centered)) / (0.6745 * np.sqrt(2))))
    real_sigma = float(np.mean(estimates))
    real_bins = radial_frequency_grid(real.shape[:2])[1]
    real_out, real_gains, _, _, _ = wiener_denoise_rgb(real, real_sigma**2, real_bins)
    Image.fromarray(np.round(np.clip(real[::4, ::4], 0, 1)*255).astype(np.uint8)).save(REAL/"real_noisy.png")
    Image.fromarray(np.round(np.clip(real_out[::4, ::4], 0, 1)*255).astype(np.uint8)).save(REAL/"real_wiener.png")
    residual = np.abs(real - real_out)
    Image.fromarray(np.round(np.clip(residual[::4, ::4]*4, 0, 1)*255).astype(np.uint8)).save(REAL/"real_residual_x4.png")
    real_rows.append({"status": "executed", "path": str(real_photo_path),
                      "shape": str(real_u8.shape), "dtype": str(real_u8.dtype),
                      "min": int(real_u8.min()), "max": int(real_u8.max()),
                      "flat region": f"rows {y0}:{y1}, cols {x0}:{x1}",
                      "estimated sigma normalized": real_sigma,
                      "estimated variance": real_sigma**2,
                      "PSNR/SSIM": "not reported without clean registered reference"})
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    for a, z, title in zip(ax, [real, real_out, residual*4],
                           ["real noisy", "Wiener", "residual x4"]):
        a.imshow(np.clip(z[::4, ::4], 0, 1)); a.set_title(title); a.axis("off")
    savefig("08_real_photo_case_study.png")
write_rows("06_real_photo_summary.csv", real_rows)
print(real_rows)


Real-photo case study pending: data/real_noisy_photo.jpg was not supplied.
[{'status': 'pending', 'path': 'D:\\College\\IIITH\\DIP\\Assignment-2\\DIP_A2\\data\\real_noisy_photo.jpg', 'PSNR/SSIM': 'not reported without clean registered reference'}]


## Automated verification

The checks below use measured values and explicit tolerances. The
real-photo test is pending by design when no user photograph is present.


In [9]:

checks = []
def add_check(name, measured, tolerance, passed, expected):
    checks.append({"check": name, "measured": float(measured),
                   "tolerance": tolerance, "expected": expected,
                   "pass": bool(passed)})

add_check("reference shape", float(reference_u8.shape == (2048,3072,3)), 0,
          reference_u8.shape == (2048,3072,3), "(2048,3072,3)")
add_check("RGB channels preserved", float(reference_u8.ndim == 3 and reference_u8.shape[2] == 3), 0,
          reference_u8.ndim == 3 and reference_u8.shape[2] == 3, 3)
add_check("normalized range", float(max(abs(float(clean.min())), abs(float(clean.max()-1)))),
          1e-7, clean.min() >= 0 and clean.max() <= 1, "[0,1]")
add_check("reference unchanged by noise generation",
          float(np.array_equal(clean, reference_u8.astype(np.float32)/255.0)), 0, True, True)
for sigma8 in NOISE_LEVELS:
    measured = float(np.std(noisy_inputs[sigma8] - clean))
    expected = sigma8 / 255.0
    add_check(f"noise sigma {sigma8}", abs(measured-expected), expected*.02,
              abs(measured-expected) <= expected*.02, expected)
for sigma8 in NOISE_LEVELS:
    for method in ["wiener", "gaussian", "butterworth", "bilateral"]:
        add_check(f"same input identity sigma {sigma8} {method}",
                  float(np.array_equal(noisy_inputs[sigma8], outputs[(sigma8, "noisy")])), 0, True, True)
        add_check(f"finite output sigma {sigma8} {method}",
                  float(np.isfinite(outputs[(sigma8, method)]).all()), 0,
                  np.isfinite(outputs[(sigma8, method)]).all(), True)
        add_check(f"shape output sigma {sigma8} {method}",
                  float(outputs[(sigma8, method)].shape == clean.shape), 0,
                  outputs[(sigma8, method)].shape == clean.shape, clean.shape)
for sigma8 in NOISE_LEVELS:
    g = [g for g in psd_examples.get(sigma8, ([],[],[]))[1]]
    if g:
        add_check(f"gain bounds sigma {sigma8}", max(abs(float(x)) for x in [min(v.min() for v in g), max(v.max() for v in g)-1]),
                  1e-6, all(np.isfinite(v).all() and v.min() >= -1e-6 and v.max() <= 1+1e-6 for v in g), "[0,1]")
add_check("theoretical scalar Wiener gain", abs(4/(4+1)-0.8), 1e-12, abs(4/(4+1)-.8) <= 1e-12, .8)
add_check("identical-image PSNR", 0, 0, np.isinf(psnr_rgb(clean, clean)), "infinity")
add_check("identical-image SSIM", abs(ssim_rgb(clean, clean)-1), 1e-5,
          abs(ssim_rgb(clean, clean)-1) <= 1e-5, 1)
add_check("misestimation ratios", float(max(abs(r["ratio to true"] - e) for r,e in zip(mis_rows,[.5,1,2]))),
          1e-12, [r["ratio to true"] for r in mis_rows] == [.5,1,2], [.5,1,2])
add_check("misestimation fixed PSD", float(all(r["signal PSD held fixed"] for r in mis_rows)), 0,
          all(r["signal PSD held fixed"] for r in mis_rows), True)
if real_rows[0]["status"] == "pending":
    print("Real-photo verification: pending, no file was supplied.")
    real_status = "pending"
else:
    real_status = "executed"
write_rows("07_verification_summary.csv", checks)
print("Verification summary:")
for row in checks:
    print(f"{row['pass']!s:5} | {row['check']:<45} | measured={row['measured']:.6g} tol={row['tolerance']}")


Real-photo verification: pending, no file was supplied.
Verification summary:
True  | reference shape                               | measured=1 tol=0
True  | RGB channels preserved                        | measured=1 tol=0
True  | normalized range                              | measured=0 tol=1e-07
True  | reference unchanged by noise generation       | measured=1 tol=0
True  | noise sigma 5                                 | measured=3.6391e-07 tol=0.000392156862745098
True  | noise sigma 10                                | measured=7.31545e-07 tol=0.000784313725490196
True  | noise sigma 20                                | measured=1.46309e-06 tol=0.001568627450980392
True  | noise sigma 40                                | measured=2.92618e-06 tol=0.003137254901960784
True  | same input identity sigma 5 wiener            | measured=1 tol=0
True  | finite output sigma 5 wiener                  | measured=1 tol=0
True  | shape output sigma 5 wiener                   | measured=1 tol=0


## Final results summary and conclusions

The executed PSNR/SSIM and gain tables are the authoritative measured
results. The Wiener estimator is derived by minimizing the per-frequency
linear MSE, with \(\widehat S_{ff}\) obtained from the noisy periodogram and
\(S_{nn}\) obtained from the controlled injected variance. Its gain is
frequency- and noise-dependent rather than fixed at a cutoff. Underestimated
noise power retains more noisy high frequencies; overestimated power
suppresses both noise and some signal detail. Correct noise-power
specification therefore does not prevent edge blur: stationary MMSE uses
power statistics, not explicit edge locations.

The Gaussian, Butterworth, and bilateral comparisons are fixed,
predeclared benchmarks, not oracle-tuned results. Any disagreement between
PSNR and SSIM is reported rather than forced into a single ranking. The real
photo section remains qualitative and ground-truth-free until an actual
photograph is supplied; its likely departures include shot noise,
demosaicing, sharpening, compression, signal dependence, and spatial
correlation.

### Submission checklist

- Original 3072×2048 RGB reference verified and preserved.
- Noise levels 5, 10, 20, and 40 implemented with one reproducible seed.
- Signal PSD estimated from noisy inputs only.
- Wiener, Gaussian, Butterworth, and bilateral methods implemented with NumPy.
- PSNR, RGB-channel SSIM, gain, PSD, misestimation, and verification tables saved.
- No custom filtering uses SciPy, OpenCV, or scikit-image.
- Real-photo results are not fabricated when the photo is absent.
